In [1]:
from pathlib import Path
import os
import random
import json
import pickle
from collections import Counter

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

/home/jamyoung/miniconda3/envs/yolov3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path("../preprocess/preprocessing")
META_PATH = ROOT / "metadata.csv"

classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}

seed = 42
model_name = "tf_efficientnetv2_m"

image_size = 300
batch_size = 8
num_workers = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("root exists:", ROOT.exists())
print("metadata exists:", META_PATH.exists())

device: cuda
root exists: True
metadata exists: True


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(seed)

In [4]:
df = pd.read_csv(META_PATH)

print(df.head())
print(df.columns)
print(df.shape)
print(df["dx"].value_counts())

required_cols = [
    "lesion_id",
    "image_id",
    "dx",
    "dx_type",
    "age",
    "sex",
    "localization",
    "dataset"
]

for col in required_cols:
    assert col in df.columns, f"Missing column: {col}"

def get_image_path(row):
    cls = row["dx"]
    image_id = row["image_id"]
    path = ROOT / cls / "enhanced" / f"{image_id}.jpg"
    if path.exists():
        return str(path)
    return None

df["image_path"] = df.apply(get_image_path, axis=1)

missing = df["image_path"].isna().sum()
print("missing images:", missing)

if missing > 0:
    missing_df = df[df["image_path"].isna()][["image_id", "dx", "image_path"]]
    display(missing_df.head())
    raise FileNotFoundError(f"{missing} images missing. Check image paths.")

df["label"] = df["dx"].map(label2idx).astype(int)

print(df[["image_id", "dx", "label", "image_path"]].head())
print(df["label"].value_counts().sort_index())

     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

        dataset  
0  vidir_modern  
1  vidir_modern  
2  vidir_modern  
3  vidir_modern  
4  vidir_modern  
Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'dataset'],
      dtype='object')
(10015, 8)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
missing images: 0
       image_id   dx  label                                         image_path
0  ISIC_0027419  bkl      2  ../preprocess/preprocessing/bkl/enhanced/ISIC_...
1  ISIC_0025030  bkl      2  ../prepr

In [5]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["dx"],
    random_state=seed
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train:", len(train_df))
print(train_df["dx"].value_counts())

print("\nval:", len(val_df))
print(val_df["dx"].value_counts())

train: 9013
dx
nv       6034
mel      1002
bkl       989
bcc       463
akiec     294
vasc      128
df        103
Name: count, dtype: int64

val: 1002
dx
nv       671
mel      111
bkl      110
bcc       51
akiec     33
vasc      14
df        12
Name: count, dtype: int64


In [6]:
train_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [24]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import timm

EFF_SOFT_META_DIR = Path("outputs_effnetv2_m_softattention_metadata_finetune_classifier")
EFF_SOFT_META_DIR.mkdir(parents=True, exist_ok=True)

num_classes = len(classes)

print("model_name:", model_name)
print("device:", device)
print("train size:", len(train_df))
print("val size:", len(val_df))
print("classes:", classes)

model_name: tf_efficientnetv2_m
device: cuda
train size: 9013
val size: 1002
classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [8]:
train_df_meta = train_df.copy()
val_df_meta = val_df.copy()

# age 缺失值：用训练集 median
age_median = train_df_meta["age"].median()

train_df_meta["age"] = train_df_meta["age"].fillna(age_median)
val_df_meta["age"] = val_df_meta["age"].fillna(age_median)

# sex/localization 缺失值：填 unknown
for col in ["sex", "localization"]:
    train_df_meta[col] = train_df_meta[col].fillna("unknown").astype(str)
    val_df_meta[col] = val_df_meta[col].fillna("unknown").astype(str)

# age 标准化
scaler = StandardScaler()
train_age = scaler.fit_transform(train_df_meta[["age"]])
val_age = scaler.transform(val_df_meta[["age"]])

# sex/localization one-hot
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

train_cat = encoder.fit_transform(train_df_meta[["sex", "localization"]])
val_cat = encoder.transform(val_df_meta[["sex", "localization"]])

# 拼接 metadata
train_meta = np.concatenate([train_age, train_cat], axis=1).astype("float32")
val_meta = np.concatenate([val_age, val_cat], axis=1).astype("float32")

metadata_dim = train_meta.shape[1]

print("metadata_dim:", metadata_dim)
print("train_meta:", train_meta.shape)
print("val_meta:", val_meta.shape)
print("encoder categories:", encoder.categories_)

metadata_dim: 19
train_meta: (9013, 19)
val_meta: (1002, 19)
encoder categories: [array(['female', 'male', 'unknown'], dtype=object), array(['abdomen', 'acral', 'back', 'chest', 'ear', 'face', 'foot',
       'genital', 'hand', 'lower extremity', 'neck', 'scalp', 'trunk',
       'unknown', 'upper extremity'], dtype=object)]


In [10]:
import json

metadata_info = {
    "metadata_dim": int(metadata_dim),
    "age_median": float(age_median),
    "sex_categories": encoder.categories_[0].tolist(),
    "localization_categories": encoder.categories_[1].tolist(),
}

with open(EFF_SOFT_META_DIR / "metadata_info.json", "w", encoding="utf-8") as f:
    json.dump(metadata_info, f, indent=2, ensure_ascii=False)

metadata_info

{'metadata_dim': 19,
 'age_median': 50.0,
 'sex_categories': ['female', 'male', 'unknown'],
 'localization_categories': ['abdomen',
  'acral',
  'back',
  'chest',
  'ear',
  'face',
  'foot',
  'genital',
  'hand',
  'lower extremity',
  'neck',
  'scalp',
  'trunk',
  'unknown',
  'upper extremity']}

In [11]:
class HAMImageMetadataDataset(Dataset):
    def __init__(self, dataframe, metadata_array, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.metadata_array = metadata_array
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        meta = torch.tensor(self.metadata_array[idx], dtype=torch.float32)
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        image_id = row["image_id"]

        return image, meta, label, image_id

In [12]:
train_dataset_meta = HAMImageMetadataDataset(
    train_df_meta,
    train_meta,
    transform=train_tfms
)

val_dataset_meta = HAMImageMetadataDataset(
    val_df_meta,
    val_meta,
    transform=val_tfms
)

train_loader_meta = DataLoader(
    train_dataset_meta,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

val_loader_meta = DataLoader(
    val_dataset_meta,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

batch = next(iter(train_loader_meta))
images, metas, labels, image_ids = batch

print("images:", images.shape)
print("metas:", metas.shape)
print("labels:", labels.shape)
print("image_ids:", image_ids[:3])

images: torch.Size([8, 3, 300, 300])
metas: torch.Size([8, 19])
labels: torch.Size([8])
image_ids: ['ISIC_0028345', 'ISIC_0031165', 'ISIC_0028873']


In [14]:
class_counts = train_df_meta["label"].value_counts().sort_index().values
N = class_counts.sum()
C = num_classes

class_weights = N / (C * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

for cls, count, weight in zip(classes, class_counts, class_weights):
    print(f"{cls:6s} count={count:5d}, weight={weight.item():.4f}")

pd.DataFrame({
    "class": classes,
    "count": class_counts,
    "weight": class_weights.numpy()
}).to_csv(EFF_SOFT_META_DIR / "class_weights.csv", index=False)

akiec  count=  294, weight=4.3795
bcc    count=  463, weight=2.7809
bkl    count=  989, weight=1.3019
df     count=  103, weight=12.5007
mel    count= 1002, weight=1.2850
nv     count= 6034, weight=0.2134
vasc   count=  128, weight=10.0592


In [15]:
class SoftAttention(nn.Module):
    """
    Soft-Attention module applied on the final CNN feature map.

    Input:
        x: [B, C, H, W]

    Output:
        out: [B, 2C, H, W]
        attn_map: [B, 1, H, W]
    """
    def __init__(self, in_channels, num_heads=16):
        super().__init__()

        self.num_heads = num_heads

        self.attention_conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=num_heads,
            kernel_size=1,
            bias=True
        )

        # gamma 初始化为 0，训练初期模型接近原始 backbone，比较稳定
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape

        attn = self.attention_conv(x)              # [B, K, H, W]
        attn = attn.view(B, self.num_heads, -1)    # [B, K, H*W]
        attn = torch.softmax(attn, dim=-1)
        attn = attn.view(B, self.num_heads, H, W)  # [B, K, H, W]

        attn_map = attn.mean(dim=1, keepdim=True)  # [B, 1, H, W]

        attended = x * attn_map
        attended = self.gamma * attended

        out = torch.cat([x, attended], dim=1)       # [B, 2C, H, W]

        return out, attn_map


class EfficientNetV2SoftAttentionMetadataClassifier(nn.Module):
    def __init__(
        self,
        metadata_dim,
        model_name="tf_efficientnetv2_m",
        num_classes=7,
        pretrained=True,
        num_heads=16
    ):
        super().__init__()

        # features_only=True：保留空间 feature map，方便加 Soft-Attention
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=True,
            out_indices=(-1,)
        )

        feature_channels = self.backbone.feature_info.channels()[-1]

        self.soft_attention = SoftAttention(
            in_channels=feature_channels,
            num_heads=num_heads
        )

        self.image_pool = nn.AdaptiveAvgPool2d(1)

        self.metadata_mlp = nn.Sequential(
            nn.Linear(metadata_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(feature_channels * 2 + 64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, num_classes)
        )

    def forward(self, image, metadata, return_attention=False):
        feature_map = self.backbone(image)[-1]          # [B, C, H, W]

        feature_map, attn_map = self.soft_attention(feature_map)

        image_feat = self.image_pool(feature_map)
        image_feat = image_feat.flatten(1)

        meta_feat = self.metadata_mlp(metadata)

        feat = torch.cat([image_feat, meta_feat], dim=1)
        logits = self.classifier(feat)

        if return_attention:
            return logits, attn_map

        return logits

In [16]:
eff_meta_model = EfficientNetV2SoftAttentionMetadataClassifier(
    metadata_dim=metadata_dim,
    model_name=model_name,
    num_classes=num_classes,
    pretrained=True,
    num_heads=16
).to(device)

criterion_meta = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer_meta = torch.optim.AdamW(
    eff_meta_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_meta = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_meta,
    mode="max",
    factor=0.5,
    patience=3
)

print("feature channels:", eff_meta_model.backbone.feature_info.channels()[-1])

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


feature channels: 512


In [17]:
batch = next(iter(train_loader_meta))
images, metas, labels, image_ids = batch

eff_meta_model.eval()

with torch.no_grad():
    images = images.to(device)
    metas = metas.to(device)
    labels = labels.to(device)

    logits, attn_map = eff_meta_model(
        images,
        metas,
        return_attention=True
    )

    loss = criterion_meta(logits, labels)

print("logits:", logits.shape)
print("attention map:", attn_map.shape)
print("loss:", loss.item())

logits: torch.Size([8, 7])
attention map: torch.Size([8, 1, 10, 10])
loss: 1.87628972530365


In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

In [19]:
def train_one_epoch_meta(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, metas, labels, _ in tqdm(loader, desc="Training EfficientNetV2 + Metadata"):
        images = images.to(device)
        metas = metas.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images, metas)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = avg_loss

    return metrics


@torch.no_grad()
def evaluate_meta(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    all_image_ids = []

    for images, metas, labels, image_ids in tqdm(loader, desc="Validating EfficientNetV2 + Metadata"):
        images = images.to(device)
        metas = metas.to(device)
        labels = labels.to(device)

        logits = model(images, metas)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        total_loss += loss.item() * images.size(0)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())
        all_image_ids.extend(list(image_ids))

    avg_loss = total_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = avg_loss

    return (
        metrics,
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
        all_image_ids
    )

In [28]:
print(f"EFF_SOFT_META_DIR = {EFF_SOFT_META_DIR}")
print(f"目录是否存在: {EFF_SOFT_META_DIR.exists()}")
print(f"完整路径: {EFF_SOFT_META_DIR.absolute()}")


EFF_SOFT_META_DIR = outputs_effnetv2_m_softattention_metadata_finetune_classifier
目录是否存在: True
完整路径: /mnt/e/BIP/Proj_x/EFFNet/outputs_effnetv2_m_softattention_metadata_finetune_classifier


In [29]:
# M12 + M13 合并版：直接训练 5 epoch

num_epochs = 5
patience = 3
best_f1 = -1.0
no_improve = 0
history = []

for epoch in range(1, num_epochs + 1):
    print("\n" + "=" * 80)
    print(f"Epoch {epoch}/{num_epochs}")
    print("=" * 80)

    # train
    train_metrics = train_one_epoch_meta(
        eff_meta_model,
        train_loader_meta,
        optimizer_meta,
        criterion_meta,
        device
    )

    # validate
    val_metrics, y_val_meta, y_pred_meta, y_prob_meta, val_image_ids_meta = evaluate_meta(
        eff_meta_model,
        val_loader_meta,
        criterion_meta,
        device
    )

    # scheduler uses validation macro-F1
    scheduler_meta.step(val_metrics["f1_macro"])

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "lr": optimizer_meta.param_groups[0]["lr"]
    }

    history.append(row)

    print(row)
    print("pred distribution:", Counter(y_pred_meta))

    # save training history every epoch
    history_df = pd.DataFrame(history)
    history_df.to_csv(EFF_SOFT_META_DIR / "training_history.csv", index=False)

    current_f1 = val_metrics["f1_macro"]

    if current_f1 > best_f1:
        best_f1 = current_f1
        no_improve = 0

        # save best checkpoint
        torch.save(
            {
                "model_state_dict": eff_meta_model.state_dict(),
                "classes": classes,
                "label2idx": label2idx,
                "idx2label": idx2label,
                "model_name": model_name,
                "image_size": image_size,
                "metadata_dim": metadata_dim,
                "age_median": age_median,
                "class_weights": class_weights,
                "architecture": "EfficientNetV2-M + Soft-Attention + Metadata",
            },
            EFF_SOFT_META_DIR / "best_effnetv2_softattention_metadata_classifier.pth"
        )

        # save best metrics
        pd.DataFrame([val_metrics]).to_csv(
            EFF_SOFT_META_DIR / "metrics.csv",
            index=False
        )

        # save predictions
        pred_df = pd.DataFrame({
            "image_id": val_image_ids_meta,
            "true_label": [idx2label[i] for i in y_val_meta],
            "pred_label": [idx2label[i] for i in y_pred_meta],
            "correct": y_val_meta == y_pred_meta
        })

        for i, cls in idx2label.items():
            pred_df[f"prob_{cls}"] = y_prob_meta[:, i]

        pred_df.to_csv(EFF_SOFT_META_DIR / "predictions.csv", index=False)

        # save per-class metrics
        report = classification_report(
            y_val_meta,
            y_pred_meta,
            target_names=classes,
            output_dict=True,
            zero_division=0
        )

        pd.DataFrame(report).transpose().to_csv(
            EFF_SOFT_META_DIR / "per_class_metrics.csv"
        )

        # save confusion matrix csv
        cm = confusion_matrix(
            y_val_meta,
            y_pred_meta,
            labels=list(range(num_classes))
        )

        pd.DataFrame(cm, index=classes, columns=classes).to_csv(
            EFF_SOFT_META_DIR / "confusion_matrix.csv"
        )

        print(
            f"Saved best EfficientNetV2 + Soft-Attention + Metadata model. "
            f"val_f1_macro={best_f1:.4f}"
        )

    else:
        no_improve += 1
        print(f"No improvement: {no_improve}/{patience}")

    if no_improve >= patience:
        print("Early stopping triggered.")
        break

print("\nTraining finished.")
print(f"Best val_f1_macro: {best_f1:.4f}")
print(f"Outputs saved to: {EFF_SOFT_META_DIR}")


Epoch 1/5


Training EfficientNetV2 + Metadata:   0%|          | 0/1127 [00:00<?, ?it/s]

Validating EfficientNetV2 + Metadata: 100%|██████████| 126/126 [00:25<00:00,  5.00it/s]


{'epoch': 1, 'train_accuracy': 0.758681903916565, 'train_balanced_accuracy': np.float64(0.7477226805480812), 'train_precision_macro': 0.579506588765952, 'train_recall_macro': 0.7477226805480812, 'train_f1_macro': 0.6432239853006055, 'train_loss': 0.7416317204874627, 'val_accuracy': 0.7954091816367266, 'val_balanced_accuracy': np.float64(0.8230009674147984), 'val_precision_macro': 0.6378012065273805, 'val_recall_macro': 0.8230009674147984, 'val_f1_macro': 0.6862815532999397, 'val_loss': 0.6531566159256889, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 571, np.int64(2): 118, np.int64(4): 110, np.int64(1): 96, np.int64(0): 52, np.int64(3): 42, np.int64(6): 13})
Saved best EfficientNetV2 + Soft-Attention + Metadata model. val_f1_macro=0.6863

Epoch 2/5


Validating EfficientNetV2 + Metadata: 100%|██████████| 126/126 [00:25<00:00,  5.01it/s]


...

{'epoch': 3, 'train_accuracy': 0.790413846665927, 'train_balanced_accuracy': np.float64(0.7933973974008969), 'train_precision_macro': 0.6495872886396433, 'train_recall_macro': 0.7933973974008969, 'train_f1_macro': 0.708596482851903, 'train_loss': 0.6260597227294398, 'val_accuracy': 0.8243512974051896, 'val_balanced_accuracy': np.float64(0.8240586370287432), 'val_precision_macro': 0.6572421771439, 'val_recall_macro': 0.8240586370287432, 'val_f1_macro': 0.7201692412336786, 'val_loss': 0.6116949585124668, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 626, np.int64(4): 111, np.int64(2): 102, np.int64(1): 70, np.int64(0): 50, np.int64(6): 24, np.int64(3): 19})
Saved best EfficientNetV2 + Soft-Attention + Metadata model. val_f1_macro=0.7202

Epoch 4/5


Validating EfficientNetV2 + Metadata: 100%|██████████| 126/126 [00:26<00:00,  4.84it/s]


{'epoch': 4, 'train_accuracy': 0.7974037501386886, 'train_balanced_accuracy': np.float64(0.8204384224759526), 'train_precision_macro': 0.6863127532559369, 'train_recall_macro': 0.8204384224759526, 'train_f1_macro': 0.7421903089637405, 'train_loss': 0.561771807065619, 'val_accuracy': 0.8293413173652695, 'val_balanced_accuracy': np.float64(0.8166784948159794), 'val_precision_macro': 0.7112043726996305, 'val_recall_macro': 0.8166784948159794, 'val_f1_macro': 0.7519521028491066, 'val_loss': 0.5537821106694475, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 624, np.int64(2): 123, np.int64(4): 94, np.int64(1): 89, np.int64(0): 42, np.int64(6): 17, np.int64(3): 13})
Saved best EfficientNetV2 + Soft-Attention + Metadata model. val_f1_macro=0.7520

Epoch 5/5


Validating EfficientNetV2 + Metadata: 100%|██████████| 126/126 [00:24<00:00,  5.04it/s]

{'epoch': 5, 'train_accuracy': 0.805836014645512, 'train_balanced_accuracy': np.float64(0.8227899194007735), 'train_precision_macro': 0.6769754512285662, 'train_recall_macro': 0.8227899194007735, 'train_f1_macro': 0.7368982305696887, 'train_loss': 0.5471687301155149, 'val_accuracy': 0.7994011976047904, 'val_balanced_accuracy': np.float64(0.8329003877702046), 'val_precision_macro': 0.6801361150062958, 'val_recall_macro': 0.8329003877702046, 'val_f1_macro': 0.7371827261702645, 'val_loss': 0.523423057283273, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 560, np.int64(4): 141, np.int64(2): 125, np.int64(1): 91, np.int64(0): 54, np.int64(6): 19, np.int64(3): 12})
No improvement: 1/3

Training finished.
Best val_f1_macro: 0.7520
Outputs saved to: outputs_effnetv2_m_softattention_metadata_finetune_classifier
